In [100]:
# Libraries
# General libraries to be used
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import mne
from mne.io import concatenate_raws, read_raw_edf
from mne.channels import make_standard_montage


# Potential Approach 1
from sklearn.ensemble import RandomForestClassifier



# This is the dataset to train the model
from mne.datasets.sleep_physionet.age import fetch_data

In [101]:
# This turns all unnecessary outputs off to make outputs easier to read.
mne.set_log_level("warning")

## Constants

In [102]:
# Group by Awake, Light Sleep (2 and 3), Deep Sleep (3 and 4), and REM (R)
annotation_event_id = {
    "Sleep stage W": 1,
    "Sleep stage 1": 2,
    "Sleep stage 2": 2,
    "Sleep stage 3": 3,
    "Sleep stage 4": 3,
    "Sleep stage R": 4,
}

# We want to combine stages 3 and 4, and 1 and 2, to add a layer of abstraction for the user, and make classification easier

event_id = {
    "Awake": 1,
    "Light Sleep": 2,
    "Deep Sleep": 3,
    "REM Sleep": 4,
}

frequency_bands = {
    "delta": [0.5, 4.5],
    "theta": [4.5, 8.5],
    "beta": [8.5, 11.5],
    "alpha": [11.5, 15.5],
    "sigma": [15.5, 30.5],
}

# tmin and tmax represent when epochs should start / finish
tmin = 0.0
tmax = 30.0 # By default, I am setting tmax as 30.0 but this may need to be changed to take into account the total size of files as it may not be cleanly divisible by 30

## Loading Data

In [103]:
def loading_data(subject_num, subject_index, recording_num):
    # Each participant has a subject number, and recording
    # Whats the difference between the subject number and index?

    [subject_files] = fetch_data(subjects=[subject_num], recording=[recording_num]) # Recording represents the night / session that is being loaded, 1st or 2nd

    raw_file = mne.io.read_raw_edf(
        subject_files[subject_index],
        stim_channel="Event marker",
        infer_types=True,
        exclude=['horizontal', 'oro-nasal', 'submental', 'rectal'], # Not all channels can be available to the user e.g. rectal temperature
        preload=True,
        verbose="error"
        )




    raw_file.filter(l_freq=0.5, h_freq=30.0, method="iir") # Basic preprocessing

    # Reading and setting the annotations
    annotations = mne.read_annotations(subject_files[recording_num])
    raw_file.set_annotations(annotations)



    return raw_file, annotations

In [104]:
def crop_annotations(annotations, raw_file):
    # Cropping 30 minutes before and after sleeping
    # This may need to change depending on the distribution of classes

    sleep_start = annotations[1]["onset"] - 30 * 60
    sleep_end = annotations[-2]["onset"] + 30 * 60

    annotations.crop(sleep_start, sleep_end)
    raw_file.set_annotations(annotations)

In [105]:
def create_events(raw_file):
    # Normally, mne.events_from_annotations returns a tuple so the underscore in "events, _" ensures that only the events is saved to the variable
    events, _ = mne.events_from_annotations(raw_file, event_id=annotation_event_id, chunk_duration=30.0)
    event_id = dict(awake=1, light_sleep=2, deep_sleep=3, rem=4)

    return events, event_id




In [106]:
def create_epochs(raw_file, events, event_id, picks):

    epochs = mne.Epochs(
        raw_file,
        events = events,
        event_id = event_id,
        tmin = tmin,
        tmax = tmax,
        proj = True,
        picks = ["Fpz-Cz", "Pz-Oz"],
        baseline = None,
        preload = True,
    )

    epochs_train = epochs.copy().crop() # figure out when to crop
    labels = epochs.events[0:len(epochs), -1]
    print(labels)

    return epochs, epochs_train, labels


## Visualising Data

In [107]:
# This is used to visualise the PSD for each stage
def psd_plot(raw_file):
    plt.close("all")
    raw_file.compute_psd(fmin=0.5, fmax=50).plot()
    plt.show()

# Main program

In [ ]:
night = 1

In [108]:
raw, annotations = loading_data(1, 0, 1)
events, event_id = create_events(raw)
epochs, epochs_train, labels = create_epochs(raw, events, event_id, picks)


/var/folders/n_/8b4bszmd5vsdmpldt1bzw7jr0000gn/T/ipykernel_51846/3180368219.py:23: RuntimeWarning: Limited 1 annotation(s) that were expanding outside the data range.
  raw_file.set_annotations(annotations)


[1 1 1 ... 1 1 1]


## Information about data types

In [109]:
raw.info

<Info | 8 non-empty values
 bads: []
 ch_names: Fpz-Cz, Pz-Oz, Event marker
 chs: 2 EEG, 1 Stimulus
 custom_ref_applied: False
 highpass: 0.5 Hz
 lowpass: 30.0 Hz
 meas_date: 1989-03-29 16:49:00 UTC
 nchan: 3
 projs: []
 sfreq: 100.0 Hz
 subject_info: <subject_info | his_id: X, sex: 2, first_name: Female, last_name: 33yr>
>

In [110]:
print(picks)

[0 1]


Do I need picks????

In [111]:
print(events)
print(len(events))

[[      0       0       1]
 [   3000       0       1]
 [   6000       0       1]
 ...
 [8397000       0       1]
 [8400000       0       1]
 [8403000       0       1]]
2802


Events is an array 3 by 2802 as there are 2802 samples

In [112]:
annotations

<Annotations | 126 segments: Sleep stage 1 (33), Sleep stage 2 (40), Sleep ...>

In [113]:
epochs

<Epochs | 2801 events (all good), 0 – 30 s (baseline off), ~128.3 MiB, data loaded,
 'awake': 1855
 'light_sleep': 671
 'deep_sleep': 105
 'rem': 170>

There are unbalanced class distributions for the different stages so some of the awake annotations should be cropped, idealy to around 671, the next highest stage
awake: 1855
deep_sleep: 105
light_sleep: 671
rem: 170

## Saving the Model

In [114]:

# Saving the model
"""
import joblib

joblib.dump(model, 'sleep_eeg_model.joblib')
"""

"\nimport joblib\n\njoblib.dump(model, 'sleep_eeg_model.joblib')\n"